# 08 - Build Improved CNN v2 Dataset (50 Classes + Dataset2 Overlap Augmentation)

This notebook builds a new dataset for **Improved CNN v2**.

## Goal
Keep the original **50-class task**, while adding extra training and validation images from **Dataset 2** for the **15 overlapping classes**.

## Output dataset
This notebook creates:

`I:\DeepLearning\ChineseHerb_Identify\data\processed\improved_cnn_v2_dataset\`

with the following structure:

```text
improved_cnn_v2_dataset/
├── train/
├── val/
├── test/
└── external_test_dataset2_overlap/
```

## Strategy
- Copy all classes from `dataset1/train`, `dataset1/val`, `dataset1/test`
- For the 15 overlap classes in Dataset 2:
  - 70% → add to `train`
  - 10% → add to `val`
  - 20% → keep in `external_test_dataset2_overlap`

## Notes
- This notebook does **not** rename original class folders
- It only copies files into a new processed dataset
- Output file names are prefixed so source is clear at a glance


In [1]:
# Imports
import shutil
import random
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
# Reproducibility
SEED = 42
random.seed(SEED)

In [3]:
# Paths
PROJECT_ROOT = Path(r"I:\DeepLearning\ChineseHerb_Identify")

# Dataset 1
DATASET1_ROOT = PROJECT_ROOT / "data" / "raw" / "herb50_dataset_1" / "split_dataset"
DATASET1_TRAIN_DIR = DATASET1_ROOT / "train"
DATASET1_VAL_DIR = DATASET1_ROOT / "val"
DATASET1_TEST_DIR = DATASET1_ROOT / "test"

# Dataset 2
DATASET2_ROOT = PROJECT_ROOT / "data" / "raw" / "herb101_dataset_2" / "original"

# Output dataset
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "improved_cnn_v2_dataset"
OUTPUT_TRAIN_DIR = OUTPUT_ROOT / "train"
OUTPUT_VAL_DIR = OUTPUT_ROOT / "val"
OUTPUT_TEST_DIR = OUTPUT_ROOT / "test"
OUTPUT_EXTERNAL_DIR = OUTPUT_ROOT / "external_test_dataset2_overlap"

# Metadata output
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET1_ROOT =", DATASET1_ROOT)
print("DATASET2_ROOT =", DATASET2_ROOT)
print("OUTPUT_ROOT   =", OUTPUT_ROOT)

DATASET1_ROOT = I:\DeepLearning\ChineseHerb_Identify\data\raw\herb50_dataset_1\split_dataset
DATASET2_ROOT = I:\DeepLearning\ChineseHerb_Identify\data\raw\herb101_dataset_2\original
OUTPUT_ROOT   = I:\DeepLearning\ChineseHerb_Identify\data\processed\improved_cnn_v2_dataset


In [4]:
# Helper functions
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_class_names(folder_path):
    if not folder_path.exists():
        return []
    return sorted([p.name for p in folder_path.iterdir() if p.is_dir()])

def list_image_files(folder_path):
    files = []
    for p in folder_path.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS:
            files.append(p)
    return sorted(files)

def ensure_class_dirs(base_dir, class_names):
    for class_name in class_names:
        (base_dir / class_name).mkdir(parents=True, exist_ok=True)

def copy_with_prefix(src_path, dst_dir, prefix):
    dst_dir.mkdir(parents=True, exist_ok=True)
    new_name = f"{prefix}{src_path.name}"
    dst_path = dst_dir / new_name
    shutil.copy2(src_path, dst_path)
    return dst_path

def copy_dataset_split(src_split_dir, dst_split_dir, prefix):
    class_names = get_class_names(src_split_dir)
    ensure_class_dirs(dst_split_dir, class_names)

    copied_records = []
    for class_name in class_names:
        src_class_dir = src_split_dir / class_name
        dst_class_dir = dst_split_dir / class_name

        image_files = list_image_files(src_class_dir)
        for img_path in image_files:
            new_path = copy_with_prefix(img_path, dst_class_dir, prefix)
            copied_records.append({
                "source_dataset": "dataset1",
                "source_split": src_split_dir.name,
                "class_name": class_name,
                "original_file_name": img_path.name,
                "new_file_name": new_path.name,
                "new_file_path": str(new_path)
            })
    return copied_records

In [5]:
# Check source paths
for p in [DATASET1_TRAIN_DIR, DATASET1_VAL_DIR, DATASET1_TEST_DIR, DATASET2_ROOT]:
    print(p, "->", p.exists())

I:\DeepLearning\ChineseHerb_Identify\data\raw\herb50_dataset_1\split_dataset\train -> True
I:\DeepLearning\ChineseHerb_Identify\data\raw\herb50_dataset_1\split_dataset\val -> True
I:\DeepLearning\ChineseHerb_Identify\data\raw\herb50_dataset_1\split_dataset\test -> True
I:\DeepLearning\ChineseHerb_Identify\data\raw\herb101_dataset_2\original -> True


In [6]:
# Find overlap classes
dataset1_classes = set(get_class_names(DATASET1_TRAIN_DIR))
dataset2_classes = set(get_class_names(DATASET2_ROOT))

overlap_classes = sorted(dataset1_classes.intersection(dataset2_classes))

print("Dataset 1 class count:", len(dataset1_classes))
print("Dataset 2 class count:", len(dataset2_classes))
print("Overlap class count  :", len(overlap_classes))
print("\nOverlap classes:")
print(overlap_classes)

overlap_df = pd.DataFrame({"overlap_class_name": overlap_classes})
display(overlap_df)

Dataset 1 class count: 50
Dataset 2 class count: 101
Overlap class count  : 15

Overlap classes:
['夏枯草', '巴戟天', '枸杞', '桃仁', '白头翁', '白扁豆', '白术', '白芍', '白茅根', '百合', '紫苑', '草果', '薏苡仁', '蝉蜕', '陈皮']


,overlap_class_name
0,夏枯草
1,巴戟天
2,枸杞
3,桃仁
4,白头翁
5,白扁豆
6,白术
7,白芍
8,白茅根
9,百合


In [7]:
# Prepare output directories
if OUTPUT_ROOT.exists():
    print("Output dataset already exists.")
    print("Delete it manually if you want a clean rebuild:")
    print(OUTPUT_ROOT)
else:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for p in [OUTPUT_TRAIN_DIR, OUTPUT_VAL_DIR, OUTPUT_TEST_DIR, OUTPUT_EXTERNAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ensure_class_dirs(OUTPUT_TRAIN_DIR, sorted(dataset1_classes))
ensure_class_dirs(OUTPUT_VAL_DIR, sorted(dataset1_classes))
ensure_class_dirs(OUTPUT_TEST_DIR, sorted(dataset1_classes))
ensure_class_dirs(OUTPUT_EXTERNAL_DIR, overlap_classes)

print("Prepared output folders.")

Prepared output folders.


In [8]:
# Copy dataset1 original splits into the new dataset
copied_train_records = copy_dataset_split(DATASET1_TRAIN_DIR, OUTPUT_TRAIN_DIR, prefix="d1_train_")
copied_val_records = copy_dataset_split(DATASET1_VAL_DIR, OUTPUT_VAL_DIR, prefix="d1_val_")
copied_test_records = copy_dataset_split(DATASET1_TEST_DIR, OUTPUT_TEST_DIR, prefix="d1_test_")

dataset1_copy_records = copied_train_records + copied_val_records + copied_test_records
dataset1_copy_df = pd.DataFrame(dataset1_copy_records)

print("Copied dataset1 records:", len(dataset1_copy_df))
display(dataset1_copy_df.head())

Copied dataset1 records: 62279


,source_dataset,source_split,class_name,original_file_name,new_file_name,new_file_path
0,dataset1,train,乌梅,乌梅_1.jpg,d1_train_乌梅_1.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
1,dataset1,train,乌梅,乌梅_100.jpg,d1_train_乌梅_100.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
2,dataset1,train,乌梅,乌梅_1000.jpg,d1_train_乌梅_1000.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
3,dataset1,train,乌梅,乌梅_1001.jpg,d1_train_乌梅_1001.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
4,dataset1,train,乌梅,乌梅_1002.jpg,d1_train_乌梅_1002.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...


In [9]:
# Split dataset2 overlap classes into train / val / external test
# Ratio:
# 70% -> train
# 10% -> val
# 20% -> external test

dataset2_split_records = []

for class_name in overlap_classes:
    class_dir = DATASET2_ROOT / class_name
    image_files = list_image_files(class_dir)

    random.shuffle(image_files)
    n = len(image_files)

    n_train = int(n * 0.70)
    n_val = int(n * 0.10)
    n_external = n - n_train - n_val

    train_files = image_files[:n_train]
    val_files = image_files[n_train:n_train+n_val]
    external_files = image_files[n_train+n_val:]

    print(f"{class_name}: total={n}, train={len(train_files)}, val={len(val_files)}, external={len(external_files)}")

    for img_path in train_files:
        dst_dir = OUTPUT_TRAIN_DIR / class_name
        new_path = copy_with_prefix(img_path, dst_dir, prefix="d2_train_")
        dataset2_split_records.append({
            "source_dataset": "dataset2",
            "target_split": "train",
            "class_name": class_name,
            "original_file_name": img_path.name,
            "new_file_name": new_path.name,
            "new_file_path": str(new_path)
        })

    for img_path in val_files:
        dst_dir = OUTPUT_VAL_DIR / class_name
        new_path = copy_with_prefix(img_path, dst_dir, prefix="d2_val_")
        dataset2_split_records.append({
            "source_dataset": "dataset2",
            "target_split": "val",
            "class_name": class_name,
            "original_file_name": img_path.name,
            "new_file_name": new_path.name,
            "new_file_path": str(new_path)
        })

    for img_path in external_files:
        dst_dir = OUTPUT_EXTERNAL_DIR / class_name
        new_path = copy_with_prefix(img_path, dst_dir, prefix="d2_external_")
        dataset2_split_records.append({
            "source_dataset": "dataset2",
            "target_split": "external_test_dataset2_overlap",
            "class_name": class_name,
            "original_file_name": img_path.name,
            "new_file_name": new_path.name,
            "new_file_path": str(new_path)
        })

dataset2_split_df = pd.DataFrame(dataset2_split_records)

print("Copied dataset2 overlap records:", len(dataset2_split_df))
display(dataset2_split_df.head())

夏枯草: total=86, train=60, val=8, external=18
巴戟天: total=91, train=63, val=9, external=19
枸杞: total=92, train=64, val=9, external=19
桃仁: total=168, train=117, val=16, external=35
白头翁: total=104, train=72, val=10, external=22
白扁豆: total=124, train=86, val=12, external=26
白术: total=124, train=86, val=12, external=26
白芍: total=122, train=85, val=12, external=25
白茅根: total=157, train=109, val=15, external=33
百合: total=99, train=69, val=9, external=21
紫苑: total=66, train=46, val=6, external=14
草果: total=109, train=76, val=10, external=23
薏苡仁: total=86, train=60, val=8, external=18
蝉蜕: total=95, train=66, val=9, external=20
陈皮: total=131, train=91, val=13, external=27
Copied dataset2 overlap records: 1654


,source_dataset,target_split,class_name,original_file_name,new_file_name,new_file_path
0,dataset2,train,夏枯草,66.jpg,d2_train_66.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
1,dataset2,train,夏枯草,62.jpg,d2_train_62.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
2,dataset2,train,夏枯草,45.jpg,d2_train_45.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
3,dataset2,train,夏枯草,37.jpg,d2_train_37.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...
4,dataset2,train,夏枯草,54.jpg,d2_train_54.jpg,I:\DeepLearning\ChineseHerb_Identify\data\proc...


In [10]:
# Count images in the new dataset
def count_split_images(split_dir):
    records = []
    class_names = get_class_names(split_dir)
    for class_name in class_names:
        class_dir = split_dir / class_name
        count = len(list_image_files(class_dir))
        records.append({
            "split": split_dir.name,
            "class_name": class_name,
            "count": count
        })
    return pd.DataFrame(records)

train_count_df = count_split_images(OUTPUT_TRAIN_DIR)
val_count_df = count_split_images(OUTPUT_VAL_DIR)
test_count_df = count_split_images(OUTPUT_TEST_DIR)
external_count_df = count_split_images(OUTPUT_EXTERNAL_DIR)

full_count_df = pd.concat([train_count_df, val_count_df, test_count_df, external_count_df], ignore_index=True)

print("Train total   :", train_count_df["count"].sum())
print("Val total     :", val_count_df["count"].sum())
print("Test total    :", test_count_df["count"].sum())
print("External total:", external_count_df["count"].sum())

display(full_count_df.head(20))

Train total   : 50971
Val total     : 6384
Test total    : 6232
External total: 346


,split,class_name,count
0,train,乌梅,976
1,train,侧柏叶,896
2,train,北沙参块,976
3,train,北沙参条,880
4,train,半夏,928
5,train,地龙,1048
6,train,夏枯草,940
7,train,大血藤,1056
8,train,小茴香,1040
9,train,巴戟天,1175


In [11]:
# Save metadata files
dataset1_copy_path = METADATA_DIR / "improved_cnn_v2_dataset_dataset1_copy_records.csv"
dataset2_split_path = METADATA_DIR / "improved_cnn_v2_dataset_dataset2_split_records.csv"
full_count_path = METADATA_DIR / "improved_cnn_v2_dataset_split_counts.csv"
overlap_path = METADATA_DIR / "improved_cnn_v2_overlap_classes.csv"

dataset1_copy_df.to_csv(dataset1_copy_path, index=False, encoding="utf-8-sig")
dataset2_split_df.to_csv(dataset2_split_path, index=False, encoding="utf-8-sig")
full_count_df.to_csv(full_count_path, index=False, encoding="utf-8-sig")
overlap_df.to_csv(overlap_path, index=False, encoding="utf-8-sig")

summary = {
    "dataset_name": "improved_cnn_v2_dataset",
    "num_dataset1_classes": len(dataset1_classes),
    "num_dataset2_classes": len(dataset2_classes),
    "num_overlap_classes": len(overlap_classes),
    "train_total_images": int(train_count_df["count"].sum()),
    "val_total_images": int(val_count_df["count"].sum()),
    "test_total_images": int(test_count_df["count"].sum()),
    "external_total_images": int(external_count_df["count"].sum()),
    "dataset2_overlap_split_ratio": {
        "train": 0.70,
        "val": 0.10,
        "external_test_dataset2_overlap": 0.20
    },
    "output_root": str(OUTPUT_ROOT)
}

summary_path = METADATA_DIR / "improved_cnn_v2_dataset_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print("Saved:")
print(dataset1_copy_path)
print(dataset2_split_path)
print(full_count_path)
print(overlap_path)
print(summary_path)

summary

Saved:
I:\DeepLearning\ChineseHerb_Identify\data\metadata\improved_cnn_v2_dataset_dataset1_copy_records.csv
I:\DeepLearning\ChineseHerb_Identify\data\metadata\improved_cnn_v2_dataset_dataset2_split_records.csv
I:\DeepLearning\ChineseHerb_Identify\data\metadata\improved_cnn_v2_dataset_split_counts.csv
I:\DeepLearning\ChineseHerb_Identify\data\metadata\improved_cnn_v2_overlap_classes.csv
I:\DeepLearning\ChineseHerb_Identify\data\metadata\improved_cnn_v2_dataset_summary.json


{'dataset_name': 'improved_cnn_v2_dataset',
 'num_dataset1_classes': 50,
 'num_dataset2_classes': 101,
 'num_overlap_classes': 15,
 'train_total_images': 50971,
 'val_total_images': 6384,
 'test_total_images': 6232,
 'external_total_images': 346,
 'dataset2_overlap_split_ratio': {'train': 0.7,
  'val': 0.1,
  'external_test_dataset2_overlap': 0.2},
 'output_root': 'I:\\DeepLearning\\ChineseHerb_Identify\\data\\processed\\improved_cnn_v2_dataset'}

In [12]:
# Final quick check
print("Output dataset root:", OUTPUT_ROOT)
print("\nExample train class folders:")
print(get_class_names(OUTPUT_TRAIN_DIR)[:10])

print("\nExample external test class folders:")
print(get_class_names(OUTPUT_EXTERNAL_DIR)[:20])

Output dataset root: I:\DeepLearning\ChineseHerb_Identify\data\processed\improved_cnn_v2_dataset

Example train class folders:
['乌梅', '侧柏叶', '北沙参块', '北沙参条', '半夏', '地龙', '夏枯草', '大血藤', '小茴香', '巴戟天']

Example external test class folders:
['夏枯草', '巴戟天', '枸杞', '桃仁', '白头翁', '白扁豆', '白术', '白芍', '白茅根', '百合', '紫苑', '草果', '薏苡仁', '蝉蜕', '陈皮']
